# Chat with Your Emails — Pipeline

Run everything end-to-end, or step by step. Stop and resume whenever.

**Kernel:** `chat-emails`

**Prerequisites:**
1. Place `credentials.json` in `data/` folder
2. Ollama running on Unraid (192.168.0.250:11434)
3. Qdrant running on Unraid (192.168.0.250:6333)

## 0. Setup & Imports

In [ ]:
import os
import sys
import json
from datetime import datetime

# Ensure project root is in path
os.chdir('/home/rapids/chat-with-your-emails')
sys.path.insert(0, '.')

from config.settings import config
from src.tracking.state import PipelineStateManager, PipelineStage
from src.tracking.display import ProgressDisplay
from rich.console import Console

console = Console()
display = ProgressDisplay(console)
state = PipelineStateManager()

print(f'Ollama: {config.ollama.base_url}')
print(f'Qdrant: {config.qdrant.host}:{config.qdrant.port}')
print(f'Models: LLM={config.models.text_llm}, VLM={config.models.vision_llm}, Embed={config.models.embedding}, Chat={config.models.chat}')
print(f'Pipeline state: {state.total_emails} emails tracked, status={state.status}')

## 1. Verify Connections

In [ ]:
import ollama
from qdrant_client import QdrantClient

# Test Ollama
try:
    client = ollama.Client(host=config.ollama.base_url)
    models = client.list()
    model_names = [m['name'] for m in models['models']]
    print(f'Ollama: OK — {len(model_names)} models loaded')
    for needed in [config.models.text_llm, config.models.vision_llm, config.models.embedding, config.models.chat]:
        found = any(needed in m for m in model_names)
        print(f'  {needed}: {"found" if found else "MISSING — pull it first"}')
except Exception as e:
    print(f'Ollama: FAILED — {e}')

# Test Qdrant
try:
    qc = QdrantClient(host=config.qdrant.host, port=config.qdrant.port)
    collections = qc.get_collections().collections
    print(f'Qdrant: OK — {len(collections)} collections')
    for c in collections:
        info = qc.get_collection(c.name)
        print(f'  {c.name}: {info.points_count} points')
except Exception as e:
    print(f'Qdrant: FAILED — {e}')

## 2. Gmail Setup (run once)

Only needed the first time. Will open a browser for OAuth consent.

Make sure `data/credentials.json` exists first!

In [ ]:
from src.ingestion.gmail_client import GmailClient

# Check credentials exist
creds_path = config.gmail.credentials_file
if os.path.exists(creds_path):
    print(f'Credentials found: {creds_path}')
else:
    print(f'MISSING: {creds_path}')
    print('Download from Google Cloud Console → APIs → Credentials → OAuth 2.0')
    print('Save as data/credentials.json')

# Authenticate (opens browser on first run)
if os.path.exists(creds_path):
    gmail = GmailClient()
    print('Gmail authenticated!')

## 3. Ingest Emails

Fetch emails from Gmail. Change `LIMIT` as needed.

In [ ]:
from src.ingestion.gmail_client import GmailClient, save_email

LIMIT = 10  # <-- Change this: how many emails to fetch
QUERY = ''  # <-- Optional: Gmail search query, e.g. 'from:bank' or 'subject:invoice'

gmail = GmailClient()
emails = gmail.fetch_emails(max_results=LIMIT, query=QUERY)
print(f'Fetched {len(emails)} emails from Gmail')

# Skip already-fetched
raw_dir = 'data/raw_emails'
existing = set()
if os.path.exists(raw_dir):
    existing = {f.replace('.json', '') for f in os.listdir(raw_dir) if f.endswith('.json')}

new_emails = [e for e in emails if e['message_id'] not in existing]
print(f'New: {len(new_emails)}, Already fetched: {len(emails) - len(new_emails)}')

# Save
state = PipelineStateManager()
subjects = {e['message_id']: e.get('subject', '') for e in new_emails}
senders = {e['message_id']: e.get('sender', '') for e in new_emails}
state.register_emails([e['message_id'] for e in new_emails], subjects=subjects, senders=senders)

for email in new_emails:
    save_email(email)
    state.set_stage(email['message_id'], PipelineStage.FETCHED)

print(f'Saved {len(new_emails)} emails to {raw_dir}/')

# Show what we got
for email in new_emails[:10]:
    print(f"  {email['subject'][:60]} — {email['sender'][:40]}")

## 4. Preprocess Emails

LLM extraction + VLM for images/docs. Each email is saved to `data/processed/` immediately.

Change `LIMIT` to control how many to process. Can stop and resume anytime.

In [ ]:
from src.preprocessing.pipeline import PreprocessingPipeline, _load_raw_emails

LIMIT = 10  # <-- Change this: how many to preprocess

emails = _load_raw_emails()
print(f'{len(emails)} raw emails available')

state = PipelineStateManager()
pipeline = PreprocessingPipeline(state_manager=state)

# Show current state
display.show_overview(state)

# Run preprocessing
processed = pipeline.run_preprocess(emails, limit=LIMIT)

## 5. Embed & Store

Chunk preprocessed emails, embed with bge-m3, store in Qdrant.

Change `LIMIT` to control how many to embed. Chat works immediately after.

In [ ]:
from src.preprocessing.pipeline import PreprocessingPipeline

LIMIT = 10  # <-- Change this: how many to embed (0 = all preprocessed)

state = PipelineStateManager()
pipeline = PreprocessingPipeline(state_manager=state)

display.show_overview(state)

embedded_count = pipeline.run_embed(limit=LIMIT)
print(f'Embedded {embedded_count} emails into Qdrant')

## 6. Chat!

Ask questions about your emails. Works with whatever is embedded in Qdrant.

In [ ]:
from src.embedding.embedder import EmailEmbedder
from src.storage.vector_store import EmailVectorStore

embedder = EmailEmbedder()
store = EmailVectorStore()
llm_client = ollama.Client(host=config.ollama.base_url)

info = store.get_collection_info()
print(f'Emails in vector store: {info["points_count"]} chunks')
print(f'Chat model: {config.models.chat}')
print('Ready!')

In [ ]:
# Ask a question!

QUESTION = "What are the most important emails I've received? Summarize the key ones."  # <-- Change this

SYSTEM_PROMPT = """You are a helpful email assistant. Answer based on the provided email context.
Be concise and accurate. If the context doesn't contain relevant info, say so.
Reference sender, date, and subject when available."""

# Search
query_embedding = embedder.embed_text(QUESTION)
results = store.search(query_embedding, limit=5)

# Build context
context_parts = []
for i, r in enumerate(results):
    context_parts.append(
        f"[Email {i+1}] From: {r['sender']} | Date: {r['date']} | Subject: {r['subject']}\n"
        f"Category: {r['category']}\n"
        f"{r['text'][:1500]}"
    )
context = "\n\n---\n\n".join(context_parts)

# Generate
response = llm_client.chat(
    model=config.models.chat,
    messages=[
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"Context from emails:\n\n{context}\n\nQuestion: {QUESTION}"},
    ],
    options={"temperature": 0.3},
)

print("\n" + "="*60)
print(f"Q: {QUESTION}")
print("="*60)
print(response["message"]["content"])
print("\n" + "-"*60)
print("Sources:")
for r in results:
    print(f"  [{r['score']:.3f}] {r['subject'][:50]} — {r['sender'][:30]}")

## 7. Pipeline Status

Check progress anytime.

In [ ]:
state = PipelineStateManager()
display.show_overview(state)

errors = state.get_failed_emails()
if errors:
    print(f'\n{len(errors)} errors:')
    display.show_errors(state)

## 8. Reset (if needed)

Clears all pipeline state. Use if you want to start fresh.

In [ ]:
# Uncomment to reset:
# state = PipelineStateManager()
# state.reset()
# print('Pipeline state cleared')